# 00 · Test de conectividad — Dataset Adult Income (OpenML)

**Objetivo:** confirmar, antes de construir el pipeline completo, que el clúster Serverless de Databricks Free Edition tiene salida a internet para descargar el dataset Adult Income desde OpenML.

A diferencia de `load_breast_cancer` (viene embebido en scikit-learn), `fetch_openml` necesita descargar datos externos. Este notebook prueba dos métodos y te dice cuál funciona en tu workspace.

Ejecuta las celdas en orden. Si la Opción A funciona, ya puedes usarla tal cual en el notebook 01. Si falla, usa la Opción B como respaldo.


## Opción A · `fetch_openml` (método preferido)

In [0]:
import time
from sklearn.datasets import fetch_openml

print("Intentando descargar Adult Income desde OpenML...")
t0 = time.time()

try:
    adult = fetch_openml(name="adult", version=2, as_frame=True)
    df_a = adult.frame
    elapsed = time.time() - t0
    print(f"✅ Descarga exitosa en {elapsed:.1f} segundos")
    print(f"Filas: {df_a.shape[0]}, Columnas: {df_a.shape[1]}")
    print(f"Columnas: {list(df_a.columns)}")
    display(df_a.head(10))
except Exception as e:
    print(f"❌ Falló la Opción A: {type(e).__name__}: {e}")
    print("Pasa a la Opción B (celda siguiente).")


## Opción B · Respaldo vía URL directa (si la Opción A falla)

Descarga el archivo crudo del repositorio UCI (sin cabecera) y le asigna nombres de columna manualmente. Ajusta la URL si tu workspace bloquea `archive.ics.uci.edu` pero permite otro dominio (por ejemplo un mirror en GitHub).

In [0]:
import pandas as pd

column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income",
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

try:
    df_b = pd.read_csv(url, header=None, names=column_names, skipinitialspace=True)
    print(f"✅ Descarga exitosa (Opción B)")
    print(f"Filas: {df_b.shape[0]}, Columnas: {df_b.shape[1]}")
    display(df_b.head(10))
except Exception as e:
    print(f"❌ Falló también la Opción B: {type(e).__name__}: {e}")
    print("Si ambas fallan, es un tema de red/egress del workspace: consulta con el profesor")
    print("o sube manualmente un CSV del dataset a un Volume de Unity Catalog y léelo desde ahí.")


## Verificación de columnas clave

Confirma que estén disponibles las columnas que usaremos como las 6 features base (`age`, `education-num`, `hours-per-week`, `capital-gain`, `capital-loss`, `fnlwgt`) y el target (`income` o `class`).


In [0]:
# Usa la variable que sí se haya descargado (df_a de la Opción A, o df_b de la Opción B)
df_test = df_a if "df_a" in dir() else df_b

expected = ["age", "education-num", "hours-per-week", "capital-gain", "capital-loss", "fnlwgt"]
found = [c for c in expected if c in df_test.columns]
missing = [c for c in expected if c not in df_test.columns]

print(f"Columnas encontradas: {found}")
if missing:
    print(f"⚠️ Columnas no encontradas con ese nombre exacto: {missing}")
    print("Revisa df_test.columns para ver el nombre real (puede variar guion vs guion bajo, mayúsculas, etc.)")
else:
    print("✅ Las 6 columnas base están disponibles.")

target_candidates = [c for c in df_test.columns if c.lower() in ("income", "class")]
print(f"Columna(s) target candidata(s): {target_candidates}")


## Cierre

Si alguna de las dos opciones funcionó y las columnas clave aparecen, ya tienes lo necesario para construir el notebook **01 · Feature Store** con Adult Income. Guarda qué opción funcionó (A o B) — la usaremos como base de la celda de carga de datos.
